# Dynamics via Trotterization

Here, we show a simple example of how to simulate dynamics using Trotterization. We first need to create an electronic Hamiltonian from a molecular geometry. Here, we use $\mathrm{C_2H_6S_2}$ (dimethyl disulfide) as an example. 

In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem
import cupynumeric as np

from trovolut.qm.hamiltonians import ElectronicHamiltonian

In [2]:
dmds_smiles = "CSSC"

# RDKit: SMILES -> mol
mol = Chem.MolFromSmiles(dmds_smiles)
mol = Chem.AddHs(mol)  # add hydrogens

# embed 3D coordinates
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

# extract XYZ as multiline string in standard XYZ format:
atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
conf = mol.GetConformer()

xyz_lines = []
xyz_lines.append(str(len(atoms)))
xyz_lines.append("dmds generated by RDKit")
for i, sym in enumerate(atoms):
    pos = conf.GetAtomPosition(i)
    xyz_lines.append(f"{sym} {pos.x:.8f} {pos.y:.8f} {pos.z:.8f}")

dmds_xyz = "\n".join(xyz_lines)

H_elec = ElectronicHamiltonian.from_xyz(
    xyz_source=dmds_xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
).to_qubits(mapping="jordan-wigner", sparse=True)


converged SCF energy = -864.654938672564


/Users/luismantilla/Desktop/UofT/2025-2/Projects/trovolut/.venv/lib/python3.13/site-packages/pyscf/lib/misc.py:1238: Warning: 
        Seeding the random number generator with a non-constant value
        inside Legate can lead to undefined behavior and/or errors when
        the program is executed with multiple ranks.
  random.seed()


We then get the terms of the Hamiltonian and decompose it for later use in Trotterization.

In [3]:
qmeta = H_elec.decompose()
hamcoeffs = H_elec.terms()

print("first 5 terms:")
for term in qmeta["terms"][:5]:
    print(term)

first 5 terms:
{'coeff': np.float64(89.88125911910353), 'pauli': 'I'}
{'coeff': np.float64(8.173752515440594), 'pauli': 'Z0'}
{'coeff': np.float64(-2.601594466867398e-07), 'pauli': 'Y0Z1Y2'}
{'coeff': np.float64(-2.601594466867398e-07), 'pauli': 'X0Z1X2'}
{'coeff': np.float64(1.675881825261904e-07), 'pauli': 'Y0Z1Z2Z3Y4'}


In [4]:
mats = [term['coef'] * term['matrix'] for term in hamcoeffs]

In [5]:
mats[0].shape

(65536, 65536)

In [7]:
type(mats[0])

scipy.sparse._csr.csr_matrix

In [6]:
len(mats)

4657

In [ ]:
np.linalg.expm(-1j * mats[0] * 0.01) 

: 

In [ ]:
import cupynumeric as np

In [ ]:
from trovolut.qm.hamsim import simulate_hamiltonian_no_simulated_precision

In [ ]:
psi0 = np.zeros(2**qmeta["n_qubits"], dtype=complex)
psi0[0] = 1.0  # initial state |000...0>
dyn = simulate_hamiltonian_no_simulated_precision(
    mats, psi0, 1, 10)

TypeError: cuPyNumeric does not support dtype=object